# Übung 05: Web Scraping & Web-APIs

**Übungsnotebook zur Vorbereitung auf LN 1 (Mo. 09.11.2026)** · Cheatsheet: **Kapitel 5 (Web Scraping & Web-APIs)** und Kapitel 11 (Checklisten)

| | |
|---|---|
| **Themen** | HTML-Aufbau, BeautifulSoup (`find`, `select`, Navigation), wiederholte Elemente & Paginierung, HTML-Tabellen, Parsing von Preisen, JSON-APIs mit Paginierung, RSS, `requests`, rechtliche/ethische Aspekte |
| **Format** | wie die Musterprüfungen AP01–AP03: Teil A Konzeptfragen · Teil B Fehleranalyse · Teil C Code schreiben |
| **Zeitbudget** | ca. 230 Minuten (Prüfungstempo: ca. 4.5 Min. pro Punkt) |
| **Lösungen** | keine – alles ist mit dem Cheatsheet bzw. den Kursunterlagen lösbar. Hinweise wie *(→ 4.5)* verweisen auf den Cheatsheet-Abschnitt. |

**Szenario:** Das fiktive Immobilienportal *WohnRegio* listet Mietwohnungen in der Region Winterthur. Die Seiten wurden gespeichert (wie in der Prüfung – es ist **kein Internetzugriff** nötig). Zusätzlich gibt es gespeicherte Antworten einer Quartier-API und einen RSS-Feed.

**So üben Sie prüfungsnah**
- Copilot/KI-Autocomplete ausschalten, wie in der Prüfung (Cheatsheet 1.1).
- Begründungen immer **mit konkretem Bezug zu den Daten** (Spaltennamen, Werte, Zahlen aus dem Output) – sonst gibt es in der Prüfung 0 Punkte.
- Verändern Sie die Ausgangsdaten nicht, sondern arbeiten Sie pro Aufgabe mit eigenen Variablen (z.B. `df_B1`, `df_C2`).
- Am Schluss: *Kernel → Restart & Run All* (Cheatsheet 11.5).

## Setup (wird nicht bewertet)

Die folgende Zelle erzeugt die Übungsdaten **synthetisch und reproduzierbar** (fixer Seed) im Unterordner neben diesem Notebook. Sie brauchen den Code nicht zu verstehen und sollen ihn **nicht verändern** – behandeln Sie die erzeugten Dateien so, als hätten Sie sie in der Prüfung erhalten. *(Natürlich dürfen Sie die Dateien im Editor öffnen und ansehen.)*

In [ ]:
# Libraries
import os
import re
import json
import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup

# Settings
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

# Get current working directory
print(os.getcwd())

In [ ]:
# ---------------------------------------------------------------
# Data generator (do not modify) -> folder 'data_uebung05'
# ---------------------------------------------------------------
import os
import json
import numpy as np
import pandas as pd

rng = np.random.default_rng(505)
D = 'data_uebung05'
for sub in ['', '/html', '/api', '/rss']:
    os.makedirs(D + sub, exist_ok=True)

locations = ['8400 Winterthur', '8404 Winterthur', '8406 Winterthur', '8408 Winterthur', '8472 Seuzach', '8352 Elsau']
loc_factor = [1.15, 1.05, 1.0, 0.95, 0.9, 0.85]
adjectives = ['Helle', 'Charmante', 'Moderne', 'Ruhige', 'Sonnige', 'Gemütliche', 'Neu renovierte', 'Grosszügige']

listings = []
for i in range(57):
    lid = f'W-{1001 + i}'
    rooms = float(rng.choice([1.5, 2.5, 2.5, 3.5, 3.5, 3.5, 4.5, 4.5, 5.5]))
    li = int(rng.integers(0, len(locations)))
    area = int(round(24 * rooms + rng.normal(8, 7)))
    rent = int(round((900 + 340 * rooms) * loc_factor[li] * rng.uniform(0.9, 1.12), -1))
    on_request = i in (8, 23, 37, 50)
    if on_request:                                    # luxury attic flats: price hidden
        rooms, area = 5.5, int(rng.integers(160, 210))
    listings.append(dict(lid=lid, rooms=rooms, area=area, rent=rent, loc=locations[li],
                         on_request=on_request, title=f'{adjectives[i % 8]} {rooms}-Zimmer-Wohnung' + (' (Attika)' if on_request else ''),
                         floor=f'{int(rng.integers(0, 6))}. OG', badge=('Neu' if rng.random() < 0.3 else None)))

def rent_text(l, i):
    if l['on_request']:
        return 'Preis auf Anfrage'
    r = l['rent']
    th = f'{r:,}'
    variant = i % 5
    if variant == 0:
        return f"CHF {th.replace(',', chr(39))}.–"
    if variant == 1:
        return f"CHF {th.replace(',', '’')}.– / Mt."
    if variant == 2:
        return f"CHF {th.replace(',', chr(160))}.–"
    if variant == 3:
        return f"CHF {th.replace(',', chr(39))}.00"
    return f"CHF {th.replace(',', chr(39))}.– / Mt."

def card_html(l, i):
    badge = f'\n      <span class="badge">{l["badge"]}</span>' if l['badge'] else ''
    return f"""    <article class="listing-card" data-id="{l['lid']}">
      <h3 class="title"><a href="/wohnung/{l['lid']}">{l['title']}</a></h3>
      <ul class="facts">
        <li class="rooms">{l['rooms']} Zimmer</li>
        <li class="area">{l['area']}&nbsp;m²</li>
        <li class="floor">{l['floor']}</li>
      </ul>
      <span class="rent">{rent_text(l, i)}</span>
      <span class="location">{l['loc']}</span>{badge}
    </article>"""

promo = """    <article class="listing-card promo">
      <h3 class="title"><a href="https://umzug.example.ch">Umzug? Jetzt Offerte einholen!</a></h3>
      <span class="badge">Anzeige</span>
    </article>"""

pages = {1: range(0, 22), 2: range(20, 42), 3: range(40, 57)}  # pagination overlaps
raw_rows = []
for p, idx in pages.items():
    cards = []
    for k, i in enumerate(idx):
        cards.append(card_html(listings[i], i))
        if k == 5:
            cards.append(promo)
        l = listings[i]
        raw_rows.append({'listing_id': l['lid'], 'title': l['title'], 'rooms_raw': f"{l['rooms']} Zimmer",
                         'area_raw': f"{l['area']}\xa0m²", 'rent_raw': rent_text(l, i), 'location_raw': l['loc']})
    nav = ' '.join(f'<a href="inserate_seite_{q}.html">{q}</a>' if q != p else f'<span class="current">{q}</span>' for q in pages)
    html = f"""<!DOCTYPE html>
<html lang="de">
<head><meta charset="utf-8"><title>WohnRegio – Mietwohnungen Region Winterthur (Seite {p})</title></head>
<body>
  <header><h1>Mietwohnungen in der Region Winterthur</h1>
    <p class="result-count">57 Inserate gefunden</p></header>
  <main class="results">
{chr(10).join(cards)}
  </main>
  <nav class="pagination">Seite: {nav}</nav>
  <footer><p>&copy; 2026 WohnRegio (fiktiv)</p></footer>
</body>
</html>
"""
    with open(f'{D}/html/inserate_seite_{p}.html', 'w', encoding='utf-8') as f:
        f.write(html)

pd.DataFrame(raw_rows).to_csv(f'{D}/inserate_roh.csv', index=False, encoding='utf-8')

# detail page of one listing with two tables
l = listings[16]
nk = 210
chf = lambda x: f'{x:,}'.replace(',', "'")
detail = f"""<!DOCTYPE html>
<html lang="de"><head><meta charset="utf-8"><title>{l['title']} – {l['lid']}</title></head>
<body>
  <h1>{l['title']}</h1>
  <p class="address">Rosenstrasse 14, {l['loc']}</p>
  <h2>Mietkosten</h2>
  <table class="data-table">
    <caption>Kosten</caption>
    <tr><th>Position</th><th>Betrag</th></tr>
    <tr><td>Nettomiete</td><td>CHF {chf(l['rent'])}.–</td></tr>
    <tr><td>Nebenkosten (akonto)</td><td>CHF {nk}.–</td></tr>
    <tr><td>Bruttomiete</td><td>CHF {chf(l['rent'] + nk)}.–</td></tr>
  </table>
  <h2>Details zum Objekt</h2>
  <table class="data-table">
    <caption>Objektdaten</caption>
    <tr><th colspan="2">Allgemein</th></tr>
    <tr><td>Objektart</td><td>Wohnung</td></tr>
    <tr><td>Zimmer</td><td>{l['rooms']}</td></tr>
    <tr><td>Wohnfläche</td><td>{l['area']}&nbsp;m²</td></tr>
    <tr><td>Etage</td><td>{l['floor']}</td></tr>
    <tr><th colspan="2">Ausstattung</th></tr>
    <tr><td>Balkon</td><td>ja</td></tr>
    <tr><td>Lift</td><td>nein</td></tr>
    <tr><td>Haustiere</td><td>auf Anfrage</td></tr>
    <tr><th colspan="2">Verfügbarkeit</th></tr>
    <tr><td>Bezug</td><td>01.12.2026</td></tr>
  </table>
  <p>Kontakt: <a href="mailto:vermietung@wohnregio.example.ch">vermietung@wohnregio.example.ch</a></p>
</body></html>
"""
with open(f'{D}/html/detail_{l["lid"]}.html', 'w', encoding='utf-8') as f:
    f.write(detail)

# neighbourhood API: saved paginated JSON responses (values as strings!)
quartiere = [('8400', 'Altstadt', 61.4, 'A', 122), ('8404', 'Oberwinterthur', 55.0, 'B', 122),
             ('8406', 'Töss', 57.9, 'B', 122), ('8408', 'Wülflingen', 53.2, 'C', 122),
             ('8472', 'Seuzach', 49.1, 'C', 101), ('8405', 'Seen', 52.7, 'B', 122)]
items = [{'plz': q[0], 'quartier': q[1], 'umwelt': {'laerm_db': f'{q[2]:.1f}'},
          'oev': {'gueteklasse': q[3]}, 'steuerfuss_pct': str(q[4])} for q in quartiere]
for page, chunk, nxt in [(1, items[:3], 'quartiere_page_2.json'), (2, items[3:], None)]:
    payload = {'meta': {'page': page, 'per_page': 3, 'total': len(items), 'next': nxt}, 'data': chunk}
    with open(f'{D}/api/quartiere_page_{page}.json', 'w', encoding='utf-8') as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

# RSS feed (saved)
news = [('Leerstand in Winterthur sinkt weiter', 'Mon, 05 Oct 2026 07:15:00 +0200'),
        ('Referenzzinssatz bleibt unverändert – Mieten stabil', 'Mon, 05 Oct 2026 11:40:00 +0200'),
        ('Neue Siedlung in Seuzach: 120 Wohnungen', 'Wed, 07 Oct 2026 09:05:00 +0200'),
        ('Mietpreise im Kanton Zürich steigen um 2.1 %', 'Thu, 08 Oct 2026 06:30:00 +0200'),
        ('Studie: Pendeln beeinflusst Wohnortwahl', 'Thu, 08 Oct 2026 14:10:00 +0200'),
        ('Wie hoch dürfen Nebenkosten sein?', 'Sat, 10 Oct 2026 08:00:00 +0200'),
        ('Stadt Winterthur plant günstige Mietwohnungen', 'Mon, 12 Oct 2026 10:20:00 +0200'),
        ('MIETRECHT: Was gilt bei Mängeln?', 'Mon, 12 Oct 2026 16:45:00 +0200')]
rss = ['<?xml version="1.0" encoding="UTF-8"?>', '<rss version="2.0"><channel>',
       '<title>Immo-News Region Winterthur (fiktiv)</title>']
for k, (t, d) in enumerate(news):
    rss.append(f'<item><title>{t}</title><link>https://news.example.ch/{k}</link><pubDate>{d}</pubDate></item>')
rss.append('</channel></rss>')
with open(f'{D}/rss/immo_news.xml', 'w', encoding='utf-8') as f:
    f.write('\n'.join(rss) + '\n')

print('Files created:')
for root, _, files in os.walk(D):
    for fn in sorted(files):
        print('  ', os.path.join(root, fn))

### Dateien im Überblick

| Datei | Inhalt |
|---|---|
| `html/inserate_seite_1.html` … `_3.html` | gespeicherte Ergebnisseiten (Paginierung). Jede Wohnung ist ein `<article class="listing-card">`. |
| `html/detail_W-1017.html` | Detailseite einer Wohnung mit zwei Tabellen |
| `inserate_roh.csv` | Rohdaten, die ein Kollege bereits gescrapt hat (Texte unverändert, **nicht** bereinigt) |
| `api/quartiere_page_1.json`, `_2.json` | gespeicherte Antworten der Quartier-API (paginiert über `meta.next`) |
| `rss/immo_news.xml` | gespeicherter RSS-Feed |

**Öffnen Sie die HTML-Dateien zuerst im Editor oder Browser** (Rechtsklick → Untersuchen), um die Struktur zu verstehen.

In [ ]:
# Raw data scraped by a colleague (do not modify df_raw)
df_raw = pd.read_csv('data_uebung05/inserate_roh.csv')
print('Number of rows and columns:', df_raw.shape)
df_raw.head()

---

<div style="background-color:#2E4057; color:white; padding:16px 24px; border-radius:6px; font-size:18px; margin-top:8px;">
&#9654;&nbsp; <strong>Ab hier beginnen die Übungsaufgaben — 56 Punkte</strong><br>
<span style="font-size:15px; font-weight:normal;">Tragen Sie Ihre Antworten direkt in die vorgesehenen Zellen ein und führen Sie alle Code-Zellen aus.</span>
</div>

---
## Teil A – Konzeptfragen (12 Punkte)

*Antworten ohne Bezug zum Portal «WohnRegio» bzw. zu den gespeicherten Dateien werden nicht gewertet.*

### A1 (4 Punkte)

Sie scrapen das Live-Portal mit einer Schleife über alle Ergebnisseiten. Nach ca. 50 Seiten erhalten Sie den Status `429`, einzelne Detailseiten liefern `403`.

a) Was bedeuten die beiden Codes, und wie passen Sie Ihren Scraper jeweils an?

b) Nennen Sie **zwei** rechtliche bzw. ethische Punkte, die Sie vor dem Scrapen von WohnRegio prüfen, und begründen Sie, warum eine offizielle API (falls vorhanden) vorzuziehen wäre. *(→ 5.1, 5.7)*

**Ihre Antwort:**

> [Hier eintragen]

### A2 (4 Punkte)

Sagen Sie für `html/inserate_seite_1.html` **ohne Ausführen** voraus, was die folgenden Ausdrücke liefern (Typ und ungefährer Inhalt bzw. Anzahl). Prüfen Sie Ihre Vorhersage danach in der Code-Zelle und erklären Sie die Abweichungen. *(→ 5.3, 5.4)*

```python
soup.find('span', class_='rent')                 # (1)
len(soup.select('article.listing-card'))          # (2)
len(soup.select('article.listing-card[data-id]')) # (3)
promo = soup.select_one('article.promo')
promo.get('data-id')                              # (4)
promo['data-id']                                  # (5)
```

**Ihre Vorhersage und Erklärung:**

> [Hier eintragen]

In [ ]:
# Code zum Überprüfen hier eintragen

### A3 (4 Punkte)

Bei 4 der 57 Inserate steht statt eines Preises «Preis auf Anfrage». Sehen Sie sich diese Inserate an (Titel, Zimmer, Fläche).

a) Welcher Fehlwert-Mechanismus (MCAR, MAR, MNAR) liegt vermutlich vor? Begründen Sie mit den Daten.

b) Ein Analyst entfernt diese Inserate und berichtet die Median-Miete pro Zimmerzahl. Für welche Zimmerzahl ist das Ergebnis verzerrt, und in welche Richtung? *(→ 9.4)*

**Ihre Antwort:**

> [Hier eintragen]

---
## Teil B – Fehleranalyse Python-Code (24 Punkte)

*Jeder Code läuft fehlerfrei durch, liefert aber ein falsches oder irreführendes Ergebnis.*

### B1 (8 Punkte)

**Ziel:** Alle Inserate der drei Ergebnisseiten in einen Data Frame mit den Spalten `listing_id`, `title`, `rent_raw`, `location_raw`, `badge` (Text oder `None`). Das Portal gibt auf jeder Seite **57 Inserate** an.

In [ ]:
# Original code (do not modify this cell)
rows = []
for page in [1, 2, 3]:
    with open(f'data_uebung05/html/inserate_seite_{page}.html', 'r', encoding='utf-8') as file:
        soup = BeautifulSoup(file.read(), 'html.parser')
    for card in soup.select('article.listing-card'):
        try:
            rows.append([card.get('data-id'),
                         card.select('h3.title')[0].get_text(strip=True),
                         card.select('span.rent')[0].get_text(strip=True),
                         card.select('span.location')[0].get_text(strip=True),
                         card.select('span.badge')[0].get_text(strip=True)])
        except:
            continue

df_B1_orig = pd.DataFrame(rows, columns=['listing_id', 'title', 'rent_raw', 'location_raw', 'badge'])
print('Anzahl Inserate:', len(df_B1_orig))
df_B1_orig.head(8)

**Aufgabe:** Finden Sie **drei** Ursachen, warum die Zeilenzahl nicht 57 ist bzw. warum Zeilen im Ergebnis nicht hingehören (Tipp: `value_counts()` auf `listing_id` und `badge`, Seiten im Editor ansehen). Schreiben Sie den korrigierten Code, der genau 57 Inserate liefert, und begründen Sie jede Änderung. *(→ 5.4, 5.5, 11.2)*

**Begründung der Änderungen:**

> [Hier eintragen]

**Korrigierter Code:**

In [ ]:
# Korrigierten Code unter den Kommentaren eintragen

# Loop over pages and cards

# Create a DataFrame and remove duplicates

# Check the number of listings

### B2 (8 Punkte)

**Ziel:** Aus `df_raw` die Nettomiete als Zahl (`rent_chf`) extrahieren und die **Median-Miete** sowie die Anzahl Inserate **ohne** Preis ausgeben. Laut Portal haben **genau 4** Inserate keinen Preis («Preis auf Anfrage»).

In [ ]:
# Original code (do not modify this cell)
df_B2_orig = df_raw.drop_duplicates(subset='listing_id').copy()
df_B2_orig['rent_chf'] = pd.to_numeric(df_B2_orig['rent_raw']
                                       .str.replace('CHF ', '')
                                       .str.replace("'", '')
                                       .str.replace('.–', ''),
                                       errors='coerce')

print('Inserate ohne Preis:', df_B2_orig['rent_chf'].isna().sum())
print('Median-Miete (CHF):', df_B2_orig['rent_chf'].median())

**Aufgabe:** Zeigen Sie mit einer geeigneten Kontrolle, welche Rohwerte nicht umgewandelt werden konnten, und erklären Sie die Ursachen. Schreiben Sie den korrigierten Code (alle Preisformate werden korrekt umgewandelt, nur «Preis auf Anfrage» wird `NaN`) und begründen Sie jede Änderung. Warum ist der ursprüngliche Median **nicht** einfach «ungenauer», sondern potenziell **systematisch verzerrt**? *(→ 2.5, 6.2, 11.2)*

**Begründung der Änderungen:**

> [Hier eintragen]

**Korrigierter Code:**

In [ ]:
# Korrigierten Code unter den Kommentaren eintragen

### B3 (8 Punkte)

**Ziel:** Mittlere Wohnfläche (m²) pro Zimmerzahl aus `df_raw`. **Jedes** Inserat hat eine Flächenangabe.

In [ ]:
# Original code (do not modify this cell)
df_B3_orig = df_raw.drop_duplicates(subset='listing_id')

# Extract numbers from the text columns
areas = [float(a.replace('m²', '').strip()) for a in df_B3_orig['area_raw']]
rooms = [float(r.split(' ')[0]) for r in df_B3_orig['rooms_raw']]
df_B3_orig['area_m2'] = pd.Series(areas)
df_B3_orig['rooms'] = pd.Series(rooms)

print('Fehlende Flächen:', df_B3_orig['area_m2'].isna().sum())
df_B3_orig.groupby('rooms')['area_m2'].agg(['count', 'mean']).round(1)

**Aufgabe:** Erklären Sie, warum fehlende Werte entstehen, obwohl jedes Inserat eine Fläche hat, und warum zusätzlich **falsche** Werte in einzelnen Zeilen stehen (Tipp: Vergleichen Sie `df_B3_orig.index` mit dem Index von `pd.Series(areas)` und prüfen Sie einige Zeilen mit `area_raw` und `area_m2` nebeneinander). Schreiben Sie den korrigierten Code und begründen Sie jede Änderung. *(→ 6.3 «Falle», 11.2)*

**Begründung der Änderungen:**

> [Hier eintragen]

**Korrigierter Code:**

In [ ]:
# Korrigierten Code unter den Kommentaren eintragen

# Extract numbers from the text columns

# Mean area per number of rooms

---
## Teil C – Python-Code schreiben (20 Punkte)

*Schreiben Sie Python-Code und begründen Sie Ihre Vorgehensweise jeweils in 1–2 Sätzen.*

### C1 (4 Punkte)

Die Detailseite `html/detail_W-1017.html` enthält zwei Tabellen. Überführen Sie die Tabelle **«Objektdaten»** in einen Data Frame `df_C1` mit den Spalten `merkmal` und `wert`. Zwischenüberschriften (z.B. «Allgemein») dürfen **nicht** im Ergebnis vorkommen; geschützte Leerzeichen (`&nbsp;`) sollen durch normale Leerzeichen ersetzt sein.

Wählen Sie die Tabelle **über ihre Beschriftung** (`<caption>`) aus – nicht über die Position. Begründen Sie, warum `soup.find('table')` hier falsch wäre und wie Sie die Zwischenüberschriften erkennen. *(→ 5.4, 5.6)*

**Begründung der Vorgehensweise:**

> [Hier eintragen]

In [ ]:
# Code hier eintragen

### C2 (4 Punkte)

Lesen Sie aus derselben Detailseite die Tabelle **«Kosten»** ein, wandeln Sie die Beträge in Zahlen um und **prüfen** Sie, ob `Nettomiete + Nebenkosten = Bruttomiete` gilt (Ausgabe `True`/`False` und die drei Beträge). Vergleichen Sie die Nettomiete zusätzlich mit dem Preis von `W-1017` auf der Ergebnisseite.

Extrahieren Sie ausserdem die **Kontakt-E-Mail-Adresse** der Seite über den `mailto:`-Link. Begründen Sie Ihre Vorgehensweise. *(→ 5.3, 5.4, 5.6)*

**Begründung der Vorgehensweise:**

> [Hier eintragen]

In [ ]:
# Code hier eintragen

### C3 (4 Punkte)

Die Quartier-API liefert ihre Antworten **seitenweise**. Lesen Sie, beginnend bei `api/quartiere_page_1.json`, so lange die nächste Seite ein, bis `meta.next` den Wert `null` (`None`) hat – **ohne** die Anzahl Seiten fest im Code vorzugeben. Erstellen Sie aus allen `data`-Einträgen einen flachen Data Frame (`pd.json_normalize`) und wandeln Sie die Kennzahlen, die als Text geliefert werden, in Zahlen um.

Verbinden Sie das Ergebnis über die **Postleitzahl** mit den (deduplizierten) Inseraten aus `df_raw` (PLZ aus `location_raw` extrahieren) und zeigen Sie mit `indicator=True`, wie viele Inserate **keinen** Treffer haben. Begründen Sie den gewählten Join-Typ und den Datentyp des Schlüssels. *(→ 3.3, 5.7, 6.8)*

**Begründung der Vorgehensweise:**

> [Hier eintragen]

In [ ]:
# Code hier eintragen

### C4 (4 Punkte)

Lesen Sie den gespeicherten RSS-Feed `rss/immo_news.xml` ein (z.B. mit `BeautifulSoup(..., 'html.parser')`, `'xml'` oder `pd.read_xml(..., xpath='.//item')`). Erstellen Sie einen Data Frame mit `title`, `published` (als `datetime`) und `link`. Geben Sie aus:

1. die Anzahl Artikel **pro Tag** (Datum ohne Uhrzeit), chronologisch sortiert,
2. alle Titel, die das Wort «Miet» in **beliebiger** Gross-/Kleinschreibung enthalten.

Begründen Sie, wie Sie das Datumsformat von `pubDate` umwandeln. *(→ 2.6, 5.7, 6.2)*

**Begründung der Vorgehensweise:**

> [Hier eintragen]

In [ ]:
# Code hier eintragen

### C5 (4 Punkte)

Schreiben Sie eine Funktion `get_json(url, params=None)`, die eine Web-API robust abfragt:

- `timeout` von 10 Sekunden,
- bei Status `429` bis zu 3 Versuche mit wachsender Pause (z.B. 1, 2, 4 Sekunden),
- bei anderen Fehlerstatus eine verständliche Exception (`raise_for_status()`),
- Rückgabe des JSON-Inhalts als Python-Objekt.

Zeigen Sie **ohne Internet**, welche URL `requests` aus `https://api.example.ch/v1/inserate` und `params={'plz': 8400, 'zimmer_min': 3.5, 'sort': 'preis'}` bauen würde (Tipp: `requests.Request('GET', url, params=...).prepare().url`).

*Optional mit Internet:* Testen Sie die Funktion mit `https://api.coinbase.com/v2/prices/BTC-CHF/spot` (aus `coinbase_example.ipynb`) und wandeln Sie den Preis in eine Zahl um. Begründen Sie, warum die Pausen nötig sind. *(→ 5.1, 5.7)*

**Begründung der Vorgehensweise:**

> [Hier eintragen]

In [ ]:
# Code hier eintragen

---
### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
# Libraries
import os, platform, socket
from platform import python_version
from datetime import datetime

# Print system information
print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
print('Python Version:', python_version())
print('-----------------------------------')